# Universal Precision Runtime (UPR) — Notebook 03
## Level 4 (Variable Precision Reconstruction) & Level 5 (WikiText Perplexity Evaluation)

---

### Objective
1. Validate **Stage 3 & Stage 4 Success Criteria** from `idea.md`.
2. Reconstruct execution models across **all target precisions (16, 14, 12, 10, 8, 6, 4, and 2 bits)** from a **single BitPlane checkpoint** (`models/bitplane_qwen`).
3. Measure numerical degradation (Weight MAE/RMSE, Logit MAE/RMSE, Cosine Similarity, KL Divergence).
4. Measure text generation Top-1 token agreement %.
5. Evaluate WikiText-2 Perplexity (PPL) across all precision levels to quantify the quality-versus-precision tradeoff.

### Step 1: Environment Setup, Dependencies & Package Deployment
Set up Hugging Face authentication, mount Google Drive, install `datasets`, and import `upr` package.

In [1]:
import os
import sys
import gc
import time
import json
import torch
import torch.nn.functional as F

# Set Hugging Face Token
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

# Mount Google Drive if in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
    DRIVE_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
except ImportError:
    print('Running in local environment.')

WORK_DIR = os.getcwd()
print(f"Active Working Directory: {WORK_DIR}")
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Automatic Bootstrap: Ensure package files exist with NaN/Inf protection
os.makedirs("upr", exist_ok=True)

with open("upr/__init__.py", "w", encoding="utf-8") as f:
    f.write('''from .bit_ops import (
    float16_to_uint16_numpy,
    uint16_to_float16_torch,
    extract_bit_plane_np,
    pack_bit_plane,
    unpack_bit_plane,
    reconstruct_tensor,
)
from .converter import convert_to_bitplanes
from .loader import BitPlaneModel
from .metrics import compute_weight_metrics

__version__ = "0.1.0"
__all__ = [
    "float16_to_uint16_numpy",
    "uint16_to_float16_torch",
    "extract_bit_plane_np",
    "pack_bit_plane",
    "unpack_bit_plane",
    "reconstruct_tensor",
    "convert_to_bitplanes",
    "BitPlaneModel",
    "compute_weight_metrics",
]
''')

with open("upr/bit_ops.py", "w", encoding="utf-8") as f:
    f.write('''import torch
import numpy as np
import gc
from typing import Tuple, Dict, Optional, Union

def float16_to_uint16_numpy(tensor: torch.Tensor) -> np.ndarray:
    np_f16 = tensor.detach().cpu().to(torch.float16).numpy()
    return np_f16.view(np.uint16)

def uint16_to_float16_torch(np_uint16: np.ndarray, device: Union[str, torch.device] = 'cpu') -> torch.Tensor:
    np_f16 = np_uint16.view(np.float16)
    tensor = torch.from_numpy(np_f16).to(device)
    if torch.isnan(tensor).any() or torch.isinf(tensor).any():
        tensor = torch.nan_to_num(tensor, nan=0.0, posinf=65504.0, neginf=-65504.0)
    return tensor

def extract_bit_plane_np(uint16_arr: np.ndarray, bit_index: int) -> np.ndarray:
    assert 0 <= bit_index <= 15, f"bit_index must be between 0 and 15, got {bit_index}"
    return ((uint16_arr >> bit_index) & 1).astype(np.uint8)

def pack_bit_plane(bit_arr: np.ndarray) -> bytes:
    flat = bit_arr.ravel()
    packed = np.packbits(flat, bitorder='big')
    return packed.tobytes()

def unpack_bit_plane(packed_bytes: bytes, num_elements: int, shape: Optional[Tuple[int, ...]] = None) -> np.ndarray:
    packed_np = np.frombuffer(packed_bytes, dtype=np.uint8)
    unpacked = np.unpackbits(packed_np, bitorder='big')[:num_elements]
    if shape is not None:
        unpacked = unpacked.reshape(shape)
    return unpacked.astype(np.uint8)

def reconstruct_tensor(planes_dict: Dict[int, bytes], selected_bits: int, original_shape: Tuple[int, ...], device: Union[str, torch.device] = 'cpu') -> torch.Tensor:
    assert 1 <= selected_bits <= 16, f"selected_bits must be between 1 and 16, got {selected_bits}"
    num_elements = int(np.prod(original_shape)) if len(original_shape) > 0 else 1
    accum = np.zeros(num_elements, dtype=np.uint32)
    start_bit = 15
    end_bit = 16 - selected_bits
    for b in range(start_bit, end_bit - 1, -1):
        if b in planes_dict:
            bits = unpack_bit_plane(planes_dict[b], num_elements)
            accum |= (bits.astype(np.uint32) << b)
            del bits
    uint16_arr = accum.astype(np.uint16).reshape(original_shape)
    del accum
    tensor = uint16_to_float16_torch(uint16_arr, device=device)
    del uint16_arr
    return tensor
''')

with open("upr/converter.py", "w", encoding="utf-8") as f:
    f.write('''import os
import json
import torch
import numpy as np
from typing import Union, Optional
from tqdm import tqdm
from transformers import AutoModelForCausalLM
from .bit_ops import float16_to_uint16_numpy, extract_bit_plane_np, pack_bit_plane

def convert_to_bitplanes(model_or_path: Union[str, torch.nn.Module], output_directory: str, torch_dtype: torch.dtype = torch.float16) -> str:
    os.makedirs(output_directory, exist_ok=True)
    tensors_dir = os.path.join(output_directory, "tensors")
    os.makedirs(tensors_dir, exist_ok=True)
    if isinstance(model_or_path, str):
        model_name = model_or_path
        print(f"Loading Hugging Face model from: {model_name}")
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch_dtype, low_cpu_mem_usage=True)
    else:
        model_name = getattr(model_or_path, "name_or_path", "custom_model")
        model = model_or_path
    state_dict = model.state_dict()
    metadata = {"model_name_or_path": model_name, "num_tensors": len(state_dict), "tensors": {}}
    print(f"Converting {len(state_dict)} tensors to bit-plane format in '{output_directory}'...")
    for idx, (tensor_name, tensor) in enumerate(tqdm(state_dict.items(), desc="BitPlane Conversion")):
        tensor_folder_name = f"tensor_{idx}"
        tensor_folder_path = os.path.join(tensors_dir, tensor_folder_name)
        os.makedirs(tensor_folder_path, exist_ok=True)
        original_shape = list(tensor.shape)
        dtype_str = str(tensor.dtype).replace("torch.", "")
        uint16_arr = float16_to_uint16_numpy(tensor)
        planes_meta = {}
        for bit_idx in range(16):
            plane_filename = f"plane{bit_idx}.bin"
            plane_path = os.path.join(tensor_folder_path, plane_filename)
            bit_arr = extract_bit_plane_np(uint16_arr, bit_idx)
            packed_bytes = pack_bit_plane(bit_arr)
            with open(plane_path, "wb") as f:
                f.write(packed_bytes)
            planes_meta[str(bit_idx)] = f"tensors/{tensor_folder_name}/{plane_filename}"
        metadata["tensors"][tensor_name] = {"shape": original_shape, "dtype": dtype_str, "numel": int(tensor.numel()), "folder": f"tensors/{tensor_folder_name}", "planes": planes_meta}
    metadata_path = os.path.join(output_directory, "metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"Successfully converted model to BitPlane format at: {output_directory}")
    return output_directory
''')

with open("upr/loader.py", "w", encoding="utf-8") as f:
    f.write('''import os\nimport json\nimport gc\nimport torch\nfrom typing import Optional, Union, Dict, Any\nfrom tqdm import tqdm\nfrom transformers import AutoModelForCausalLM, AutoConfig\nfrom .bit_ops import reconstruct_tensor\n\nclass BitPlaneModel:\n    @classmethod\n    def load_reconstructed_state_dict(cls, bitplane_directory: str, bits: int = 16, device: Union[str, torch.device] = 'cpu') -> Dict[str, torch.Tensor]:\n        assert 1 <= bits <= 16, f"bits must be between 1 and 16, got {bits}"\n        metadata_path = os.path.join(bitplane_directory, "metadata.json")\n        if not os.path.exists(metadata_path):\n            raise FileNotFoundError(f"metadata.json not found in '{bitplane_directory}'")\n        with open(metadata_path, "r", encoding="utf-8") as f:\n            metadata = json.load(f)\n        reconstructed_state_dict = {}\n        tensors_meta = metadata["tensors"]\n        start_bit = 15\n        end_bit = 16 - bits\n        for idx, (tensor_name, info) in enumerate(tqdm(tensors_meta.items(), desc=f"Reconstructing ({bits}-bit)")):\n            original_shape = tuple(info["shape"])\n            planes_dict = {}\n            for b in range(start_bit, end_bit - 1, -1):\n                plane_rel_path = info["planes"][str(b)]\n                plane_full_path = os.path.join(bitplane_directory, plane_rel_path)\n                if os.path.exists(plane_full_path):\n                    with open(plane_full_path, "rb") as pf:\n                        planes_dict[b] = pf.read()\n            recon_tensor = reconstruct_tensor(planes_dict=planes_dict, selected_bits=bits, original_shape=original_shape, device=device)\n            del planes_dict\n            reconstructed_state_dict[tensor_name] = recon_tensor\n            if idx % 50 == 0:\n                gc.collect()\n        gc.collect()\n        return reconstructed_state_dict\n\n    @classmethod\n    def from_pretrained(cls, bitplane_directory: str, bits: int = 16, base_model_id: Optional[str] = None, device_map: Optional[Union[str, Dict[str, Any]]] = None, torch_dtype: torch.dtype = torch.float16, **kwargs) -> torch.nn.Module:\n        metadata_path = os.path.join(bitplane_directory, "metadata.json")\n        with open(metadata_path, "r", encoding="utf-8") as f:\n            metadata = json.load(f)\n        model_name = base_model_id or metadata.get("model_name_or_path")\n        print(f"Instantiating model base architecture '{model_name}' for precision bits={bits}...")\n        config = AutoConfig.from_pretrained(model_name)\n        model = AutoModelForCausalLM.from_config(config, torch_dtype=torch_dtype)\n        state_dict = cls.load_reconstructed_state_dict(bitplane_directory=bitplane_directory, bits=bits, device='cpu')\n        model.load_state_dict(state_dict, strict=True)\n        del state_dict\n        gc.collect()\n        if device_map is not None:\n            model = model.to(device_map)\n        return model\n''')

with open("upr/metrics.py", "w", encoding="utf-8") as f:
    f.write('''import torch\nimport numpy as np\nfrom typing import Dict, Any\n\ndef compute_weight_metrics(original: torch.Tensor, reconstructed: torch.Tensor) -> Dict[str, Any]:\n    orig_f32 = original.detach().cpu().to(torch.float32)\n    recon_f32 = reconstructed.detach().cpu().to(torch.float32)\n    is_exact = bool(torch.equal(original.detach().cpu(), reconstructed.detach().cpu()))\n    diff = torch.abs(orig_f32 - recon_f32)\n    mae = float(diff.mean().item())\n    rmse = float(torch.sqrt(torch.mean((orig_f32 - recon_f32) ** 2)).item())\n    max_error = float(diff.max().item())\n    orig_flat = orig_f32.view(-1)\n    recon_flat = recon_f32.view(-1)\n    norm_orig = torch.norm(orig_flat)\n    norm_recon = torch.norm(recon_flat)\n    if norm_orig == 0 or norm_recon == 0:\n        cos_sim = 1.0 if norm_orig == norm_recon else 0.0\n    else:\n        cos_sim = float((torch.dot(orig_flat, recon_flat) / (norm_orig * norm_recon)).item())\n    return {"exact_match": is_exact, "mae": mae, "rmse": rmse, "max_error": max_error, "cosine_similarity": cos_sim, "num_elements": int(orig_f32.numel())}\n''')

!pip install -q transformers accelerate huggingface_hub torch numpy tqdm datasets

from huggingface_hub import login
login(token=HF_TOKEN)

import upr
print(f'UPR active. PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
Active Working Directory: /content/drive/MyDrive/UniversalPrecisionRuntime


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


UPR active. PyTorch: 2.11.0+cu128, CUDA available: True


### Step 2: Perplexity & Reference Baseline Benchmark Setup
We load WikiText-2 validation dataset and evaluate the baseline FP16 model to establish ground-truth Perplexity and reference outputs.

In [2]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3.5-0.8B'
BITPLANE_DIR = 'models/bitplane_qwen'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

# Load WikiText-2 test dataset
print('Loading WikiText-2 dataset...')
try:
    test_dataset = load_dataset('salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
except Exception:
    test_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', trust_remote_code=True)

text_samples = [t for t in test_dataset['text'] if len(t.strip()) > 100][:32]
encodings = tokenizer('\n\n'.join(text_samples), return_tensors='pt')

seq_len = 512
input_ids = encodings.input_ids[:, :seq_len * 4].to(DEVICE)

def evaluate_perplexity(model, input_ids, seq_len=512):
    model.eval()
    nlls = []
    total_len = input_ids.size(1)
    for i in range(0, total_len, seq_len):
        end_loc = min(i + seq_len, total_len)
        if end_loc - i < 64:
            continue
        trg_len = end_loc - i
        chunk_ids = input_ids[:, i:end_loc]
        target_ids = chunk_ids.clone()
        
        with torch.no_grad():
            try:
                outputs = model(chunk_ids, labels=target_ids)
                loss = outputs.loss
                if torch.isnan(loss) or torch.isinf(loss):
                    return 9999.0
                neg_log_likelihood = loss * trg_len
                nlls.append(neg_log_likelihood)
            except Exception:
                return 9999.0
            
    if not nlls:
        return 9999.0
    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    val = float(ppl.item())
    return val if not (torch.isnan(ppl) or torch.isinf(ppl)) else 9999.0

print(f'Evaluating FP16 Baseline Model on {DEVICE}...')
orig_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16
).to(DEVICE)
orig_model.eval()

# Store lightweight reference logits and output text
eval_prompt = "Universal Precision Runtime allows dynamic precision reconstruction from a single bit-plane checkpoint."
prompt_inputs = tokenizer(eval_prompt, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    orig_logits = orig_model(**prompt_inputs).logits.detach().cpu().to(torch.float32)
    gen_prompt = "The key advantage of a single bit-plane representation is"
    gen_inputs = tokenizer(gen_prompt, return_tensors='pt').to(DEVICE)
    orig_output_ids = orig_model.generate(**gen_inputs, max_new_tokens=40, do_sample=False).detach().cpu()

ppl_baseline = evaluate_perplexity(orig_model, input_ids, seq_len=seq_len)
orig_text = tokenizer.decode(orig_output_ids[0], skip_special_tokens=True)

print('='*60)
print(f'FP16 BASELINE RESULT | Perplexity (PPL): {ppl_baseline:.4f}')
print('='*60)

del orig_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleared baseline model and freed memory.')

Loading WikiText-2 dataset...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Evaluating FP16 Baseline Model on cuda...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

FP16 BASELINE RESULT | Perplexity (PPL): 24.0684
Cleared baseline model and freed memory.


### Step 3: Single-Pass Variable Precision Sweep (16, 14, 12, 10, 8, 6, 4, and 2 Bits)
Single-pass reconstruction loop with zero-leak buffer deletion per precision level.

In [3]:
from transformers import AutoConfig

precisions = [16, 14, 12, 10, 8, 6, 4, 2]
sweep_results = []
os.makedirs('results', exist_ok=True)

config = AutoConfig.from_pretrained(MODEL_ID)

print(f"{'Bits':<6} | {'Recon Time':<12} | {'Logit CosSim':<14} | {'Top-1 Acc':<10} | {'Perplexity':<12}")
print("-" * 70)

for bits in precisions:
    t0 = time.time()
    # 1. Reconstruct state dict ONCE with garbage-collected bit buffers
    recon_state_dict = upr.BitPlaneModel.load_reconstructed_state_dict(BITPLANE_DIR, bits=bits, device='cpu')
    t_recon = time.time() - t0
    
    # 2. Instantiate HF model structure and load state dict directly
    recon_model = AutoModelForCausalLM.from_config(config, torch_dtype=torch.float16)
    recon_model.load_state_dict(recon_state_dict, strict=True)
    del recon_state_dict # Free state dict immediately from RAM!
    gc.collect()
    
    recon_model = recon_model.to(DEVICE)
    recon_model.eval()
    
    # 3. Forward Logits & Text Generation with exception safety
    try:
        with torch.no_grad():
            recon_logits = recon_model(**prompt_inputs).logits.detach().cpu().to(torch.float32)
            recon_output_ids = recon_model.generate(**gen_inputs, max_new_tokens=40, do_sample=False).detach().cpu()
            
        flat_o = orig_logits.view(-1)
        flat_r = recon_logits.view(-1)
        cos_sim = float((torch.dot(flat_o, flat_r) / (torch.norm(flat_o) * torch.norm(flat_r))).item())
        
        token_matches = (orig_output_ids == recon_output_ids).sum().item()
        total_gen_tokens = orig_output_ids.numel()
        token_acc = (token_matches / total_gen_tokens) * 100
    except Exception:
        cos_sim = 0.0
        token_acc = 0.0
    
    # 4. Perplexity
    ppl = evaluate_perplexity(recon_model, input_ids, seq_len=seq_len)
    
    # Clean up model from VRAM
    del recon_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    res_item = {
        'bits': bits,
        'reconstruction_time_sec': t_recon,
        'logit_cosine_similarity': cos_sim,
        'top1_token_accuracy_pct': token_acc,
        'perplexity': ppl
    }
    sweep_results.append(res_item)
    
    # Save individual precision json file
    with open(f'results/{bits}bit.json', 'w') as f:
        json.dump(res_item, f, indent=2)
        
    print(f"{bits:<6} | {t_recon:<12.3f} | {cos_sim:<14.6f} | {token_acc:<9.2f}% | {ppl:<12.4f}")

Bits   | Recon Time   | Logit CosSim   | Top-1 Acc  | Perplexity  
----------------------------------------------------------------------


Reconstructing (16-bit): 100%|██████████| 321/321 [01:26<00:00,  3.71it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


16     | 86.746       | 1.000098       | 100.00   % | 24.0684     


Reconstructing (14-bit): 100%|██████████| 321/321 [01:11<00:00,  4.49it/s]


14     | 71.768       | 1.000090       | 100.00   % | 24.0577     


...

6      | 31.709       | 0.661867       | 22.00    % | 1354.7736   


Reconstructing (4-bit): 100%|██████████| 321/321 [00:23<00:00, 13.78it/s]


4      | 23.543       | 0.076273       | 20.00    % | 101262.2344 


Reconstructing (2-bit): 100%|██████████| 321/321 [00:18<00:00, 17.15it/s]


2      | 18.958       | nan            | 20.00    % | 248319.6250 


### Step 4: Summary & Export Artifacts
Save overall variable precision sweep summary to `results/variable_precision_summary.json`.

In [4]:
summary_data = {
    'baseline_model': MODEL_ID,
    'baseline_perplexity': ppl_baseline,
    'precision_sweep': sweep_results
}

with open('results/variable_precision_summary.json', 'w') as f:
    json.dump(summary_data, f, indent=2)

print('\n' + '='*70)
print('VARIABLE PRECISION SWEEP COMPLETED SUCCESSFULLY!')
print('Saved results to results/variable_precision_summary.json & results/{N}bit.json')
print('='*70)


VARIABLE PRECISION SWEEP COMPLETED SUCCESSFULLY!
Saved results to results/variable_precision_summary.json & results/{N}bit.json
